# MIST precision memory diagnostics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aecelaya/MIST/blob/main/benchmarks/mist_precision_memory.ipynb)

Why does FP16 take more GPU memory than BF16 on some GPUs? On an A100, FP16
training peaked at 0.80 of FP32's memory, against 0.57 for BF16; on a T4, FP16
took 0.63. This trains `nnunet-plain` at LiTS-like settings (128 × 128 × 128,
batch 2) with MIST's loss in each precision, on random data, changing one thing
at a time: cuDNN's algorithm search (`cudnn.benchmark`), compiling, and where the
loss is computed. No dataset is needed; it takes about 10 minutes on an A100.

Every measurement runs in its own process, so each starts with an empty GPU and
its own peak memory. Run it on a GPU runtime, then download the CSV at the end.

## Setup

In [ ]:
# Change MIST_REF to test a branch or commit.
MIST_REF = "main"
!pip install -q "mist-medical @ git+https://github.com/aecelaya/MIST.git@{MIST_REF}"

In [ ]:
import json
import os
import platform
import subprocess
import sys
import warnings
from pathlib import Path

import pandas as pd
import torch

from mist.utils import hardware

warnings.filterwarnings("ignore")
GIB = 1024**3

# Set MIST_PRECISION_DRY_RUN to test this notebook's code on a CPU at a tiny size.
# CPU timings say nothing about a GPU.
DRY_RUN = bool(os.environ.get("MIST_PRECISION_DRY_RUN"))
if not (torch.cuda.is_available() or DRY_RUN):
    raise RuntimeError("No GPU found. Choose one under Runtime > Change runtime type.")

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "dry-run"
GPU_MEMORY = (
    torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else 16 * GIB
)
HAS_BF16 = hardware.bf16_supported()
OUTPUT_CSV = f"mist_precision_memory_{GPU_NAME.replace(' ', '_').replace('/', '-')}.csv"

ARCHITECTURE = "nnunet-plain"
MODEL_KWARGS = {
    "in_channels": 1,
    "out_channels": 3,
    "patch_size": [128, 128, 128],
    "target_spacing": [0.767578125, 0.767578125, 1.0],
}
BATCH = 2
MEMORY_STEPS = 12  # Training steps per memory run, after 3 untimed ones.

if DRY_RUN:
    MODEL_KWARGS["patch_size"] = [32, 32, 32]
    MEMORY_STEPS = 2

rows = []


def record(section, **values):
    rows.append({"gpu": GPU_NAME, "section": section, **values})
    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)


print(f"GPU: {GPU_NAME}, {GPU_MEMORY / GIB:.1f} GiB; BF16 hardware: {HAS_BF16}")
print(
    f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}, cuDNN "
    f"{torch.backends.cudnn.version()}, Python {platform.python_version()}"
)
print(f"Results go to {OUTPUT_CSV}")

## Where FP16's extra memory goes

Each run trains `nnunet-plain` as `Patch3DTrainer` does: the network under
autocast, MIST's Dice + cross-entropy loss with deep supervision, gradient scaling
in FP16 (the loss computed in FP32 from the network's outputs), clipping, and
AdamW. *Reserved* is what PyTorch holds from the GPU, the number training runs out
of; *allocated* is what tensors actually use at the peak. A gap between FP16 and
BF16 that disappears with `cudnn.benchmark` off points at the convolution
algorithms cuDNN picks and their scratch memory.

In [ ]:
MEMORY_WORKER = r"""
import json
import sys
import time

import torch

import mist.models  # noqa: F401
from mist.loss_functions.deep_supervision_wrapper import DeepSupervisionLoss
from mist.loss_functions.loss_registry import get_loss
from mist.models.model_registry import get_model_from_registry
from mist.utils import hardware

AMP = {"fp32": False, "bf16": True, "fp16": "fp16"}


def run(job):
    hardware.tune_gpu_math()
    torch.backends.cudnn.benchmark = job["benchmark"]
    cuda = torch.cuda.is_available()
    device = torch.device("cuda" if cuda else "cpu")
    model = get_model_from_registry(job["architecture"], **job["model_kwargs"]).to(device)
    if job["compiled"]:
        model.compile()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05, eps=1e-8)
    criterion = DeepSupervisionLoss(get_loss("dice_ce")())
    amp = AMP[job["precision"]]
    fp16 = amp == "fp16"
    scaler = torch.amp.GradScaler(device.type, enabled=fp16)
    patch = job["model_kwargs"]["patch_size"]
    channels = job["model_kwargs"]["in_channels"]
    classes = job["model_kwargs"]["out_channels"]
    image = torch.randn(job["batch"], channels, *patch, device=device)
    label = torch.randint(0, classes, (job["batch"], 1, *patch), device=device)

    def step():
        optimizer.zero_grad()
        with hardware.autocast_context(amp):
            output = model(image)
        prediction = output["prediction"]
        extras = output.get("deep_supervision")
        if fp16 and job["loss_in_fp32"]:
            prediction = prediction.float()
            extras = tuple(e.float() for e in extras) if extras is not None else None
        loss_amp = False if (fp16 and job["loss_in_fp32"]) else amp
        with hardware.autocast_context(loss_amp):
            loss = criterion(y_true=label, y_pred=prediction, y_supervision=extras,
                             alpha=None, dtm=None)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        if cuda:
            torch.cuda.synchronize()

    for _ in range(3):
        step()
    times = []
    for _ in range(job["steps"]):
        started = time.perf_counter()
        step()
        times.append(time.perf_counter() - started)
    times.sort()
    result = {"seconds_per_step": times[len(times) // 2]}
    if cuda:
        result["peak_reserved_gib"] = torch.cuda.max_memory_reserved() / 1024**3
        result["peak_allocated_gib"] = torch.cuda.max_memory_allocated() / 1024**3
    return result


if __name__ == "__main__":
    job = json.loads(sys.argv[1])
    try:
        print("RESULT " + json.dumps({"status": "ok", **run(job)}), flush=True)
    except Exception as error:
        message = f"{type(error).__name__}: {error}"[-600:]
        print("RESULT " + json.dumps({"status": "error", "error": message}), flush=True)
"""
Path("memory_worker.py").write_text(MEMORY_WORKER)


def memory_run(precision, benchmark=True, compiled=True, loss_in_fp32=True):
    job = {
        "architecture": ARCHITECTURE,
        "model_kwargs": MODEL_KWARGS,
        "batch": BATCH,
        "precision": precision,
        "benchmark": benchmark,
        "compiled": compiled and not DRY_RUN,
        "loss_in_fp32": loss_in_fp32,
        "steps": MEMORY_STEPS,
    }
    done = subprocess.run(
        [sys.executable, "memory_worker.py", json.dumps(job)],
        capture_output=True,
        text=True,
        timeout=1800,
        check=False,
    )
    lines = [line for line in done.stdout.splitlines() if line.startswith("RESULT ")]
    if lines:
        result = json.loads(lines[-1][len("RESULT ") :])
    else:
        # -9 is the system killing it, usually for running out of memory.
        result = {"status": f"exited with code {done.returncode}"}
    settings = {
        "precision": precision,
        "benchmark": benchmark,
        "compiled": job["compiled"],
        "loss_in_fp32": loss_in_fp32 if precision == "fp16" else "",
    }
    record("memory", **settings, **result)
    if result["status"] == "ok":
        memory = (
            f"reserved {result['peak_reserved_gib']:.2f} GiB, allocated "
            f"{result['peak_allocated_gib']:.2f} GiB"
            if "peak_reserved_gib" in result
            else "no GPU"
        )
        print(f"{settings}: {result['seconds_per_step']:.3f} s/step, {memory}")
    else:
        print(f"{settings}: {result['status']} {result.get('error', '')}")


precisions = (
    ["fp32"] + (["bf16"] if HAS_BF16 else []) + (["fp16"] if torch.cuda.is_available() else [])
)
reduced = [p for p in precisions if p != "fp32"]
# As training runs: compiled, with cuDNN's algorithm search.
for precision in precisions:
    memory_run(precision)
# One change at a time, for BF16 and FP16.
for precision in reduced:
    memory_run(precision, benchmark=False)
    memory_run(precision, compiled=False)
if "fp16" in precisions:
    memory_run("fp16", loss_in_fp32=False)  # The loss under FP16 autocast, as BF16's is.

## Results

In [ ]:
results = pd.DataFrame(rows)
memory = results[(results["section"] == "memory") & (results["status"] == "ok")]
if len(memory) and "peak_reserved_gib" in memory:
    fp32 = memory[(memory["precision"] == "fp32")]["peak_reserved_gib"]
    if len(fp32):
        memory = memory.assign(share_of_fp32=memory["peak_reserved_gib"] / fp32.iloc[0])
    columns = [
        c
        for c in [
            "precision",
            "benchmark",
            "compiled",
            "loss_in_fp32",
            "seconds_per_step",
            "peak_reserved_gib",
            "peak_allocated_gib",
            "share_of_fp32",
        ]
        if c in memory
    ]
    print("Memory runs")
    print(memory[columns].round(3).to_string(index=False))

## Save the results

Downloads the CSV. If the download doesn't start, it's in the Files panel on the
left.

In [ ]:
print(f"Saved {OUTPUT_CSV}")
try:
    from google.colab import files

    files.download(OUTPUT_CSV)
except ImportError:
    pass